# 🚀 SparseChron — 4D Interactive Viewer (Official Retrained Model)

Streamlined, zero-bloat demo runner. Pre-configured for the **Official Retrained Model** (PSNR 29.17 / SSIM 0.945 / LPIPS 0.083) with pristine, stable colors and zero artifacting.

### Requirements
- **Session options:** Accelerator = **GPU T4**, Internet = **ON**
- **Total setup time:** ~30 seconds (ZERO compilation, pre-cached CUDA kernels, Cloudflare tunnel)

Run the 3 cells below in order.

In [ ]:
# 1. Clone repo + install dependencies (INSTANT ~15s, ZERO COMPILE)
import os, subprocess, sys

REPO = "/kaggle/working/SparseChron"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/rajhodedara/SparseChron.git", REPO], check=True)
else:
    print("Repo already cloned, skipping.")

%cd /kaggle/working/SparseChron

# Install gsplat wheel and requirements
WHEEL = "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat-1.5.3-py3-none-any.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", WHEEL])
!pip install -q -r requirements.txt

# Unpack precompiled CUDA kernels directly into PyTorch JIT cache
print("Fetching precompiled gsplat CUDA kernels (0s compile time)...")
!mkdir -p /root/.cache/torch_extensions
!curl -sL https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat_cache.tar.gz | tar -xz -C /root/.cache/torch_extensions

# Touch cache timestamps to future so Ninja never rebuilds
for root, dirs, files in os.walk('/root/.cache/torch_extensions'):
    for f in files:
        try:
            os.utime(os.path.join(root, f), (2000000000, 2000000000))
        except Exception:
            pass

# Instant load precompiled C++ extension directly
from torch.utils.cpp_extension import _get_build_directory, _import_module_from_library
build_dir = _get_build_directory("gsplat_cuda", verbose=False)
_C = _import_module_from_library("gsplat_cuda", build_dir, is_python_module=True)
sys.modules["gsplat.csrc"] = _C
import gsplat.cuda._backend as _b
_b._C = _C

print("\n🚀 Environment ready! gsplat CUDA kernels hot and loaded.")


In [ ]:
# 2. Download Official Model (Pristine Colors, PSNR 29.17)
import os, urllib.request

# Ensure on main branch with latest color shader fixes
!cd /kaggle/working/SparseChron && git checkout main

os.makedirs("/kaggle/working/outputs_mutant", exist_ok=True)
BASE = "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints"

for fname in ["checkpoint_retrained_30000.ckpt", "scene_transform.json"]:
    dst = "/kaggle/working/outputs_mutant/" + ("checkpoint_30000.ckpt" if fname.startswith("checkpoint_") else fname)
    print(f"Downloading {fname}...")
    urllib.request.urlretrieve(f"{BASE}/{fname}", dst)
    print(f"  -> {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")

import torch
state = torch.load("/kaggle/working/outputs_mutant/checkpoint_30000.ckpt", map_location="cpu", weights_only=False)
print(f"\nIteration: {state.get('iteration', 'n/a')} | Gaussians: {state['model_state_dict']['_positions'].shape[0]:,}")
print("Official model loaded & ready! Colors verified.")


In [ ]:
# 3. Launch 4D Viewer (Viser) + Cloudflare Tunnel (NO PASSWORDS, Clean Colors)
import os, subprocess, time, socket, glob

# 1. Clean up old processes
!pkill -9 -f interactive_viewer; !pkill -9 -f cloudflared; !pkill -9 -f localtunnel
time.sleep(1)

# 2. Download Cloudflare binary if not present (~2s)
if not os.path.exists("/kaggle/working/cloudflared"):
    print("Downloading Cloudflare tunnel binary...")
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

# 3. Launch 4D viewer with the official model
CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_30000.ckpt"
print(f"Booting 4D viewer with model: {CKPT_PATH}")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.Popen(
    ["python", "/kaggle/working/SparseChron/scripts/interactive_viewer.py", "--checkpoint", CKPT_PATH, "--port", "8080"],
    env=env
)

# 4. Wait for Viser to bind to port 8080
print("Loading model into GPU VRAM (waiting for port 8080)...")
for _ in range(40):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 8080)) == 0:
            print("\n✅ Viser 4D Viewer is LIVE on port 8080!")
            break
    time.sleep(1)

# 5. Launch Cloudflare Tunnel (prints a clean https://*.trycloudflare.com link)
print("Opening Cloudflare Tunnel (NO PASSWORDS NEEDED)...\n")
!/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8080
